In [ ]:
# Cell 0
import os
from pathlib import Path
import math
import random
from tqdm import tqdm

import cv2
import yaml
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# Cell 1
def xywh_to_xyxy(boxes):
    # boxes: (N,4) x_center,y_center,w,h normalized [0-1]
    x_c, y_c, w, h = boxes.T
    x1 = x_c - w/2
    y1 = y_c - h/2
    x2 = x_c + w/2
    y2 = y_c + h/2
    return np.stack([x1,y1,x2,y2],axis=1)

def iou_xyxy(a, b):
    # a: (4,) x1,y1,x2,y2 ; b: (N,4)
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    inter_w = max(0, x2-x1); inter_h = max(0, y2-y1)
    inter = inter_w * inter_h
    area_a = (a[2]-a[0])*(a[3]-a[1])
    area_b = (b[:,2]-b[:,0])*(b[:,3]-b[:,1])
    union = area_a + area_b - inter
    return inter / (union + 1e-9)

# Simple average precision @ IoU=0.5 evaluator (non-optimal but useful)
def voc_ap(rec, prec):
    # 11-point interp deprecated; do trapezoid
    mrec = np.concatenate(([0.], rec, [1.]))
    mpre = np.concatenate(([0.], prec, [0.]))
    for i in range(mpre.size-2, -1, -1):
        mpre[i] = np.maximum(mpre[i], mpre[i+1])
    i = np.where(mrec[1:] != mrec[:-1])[0]
    ap = np.sum((mrec[i+1]-mrec[i]) * mpre[i+1])
    return ap


In [ ]:
# Cell 2
def make_pseudocolor(bgr):
    # bgr: uint8 HxWx3, returns uint8 HxWx3
    # Example: convert to LAB and enhance L channel contrast, then remap channels
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    L, a, b = cv2.split(lab)
    L = cv2.equalizeHist(L)
    lab_eq = cv2.merge([L,a,b])
    rgb = cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)
    # optional color jitter / stretching
    return rgb

def compute_hha(depth):
    # depth: single-channel float32 (meters or normalized) shape HxW
    # We produce 3 channels: pseudo-height, inv_depth (log), normals magnitude/angles
    # If depth values are normalized [0..1], convert to meters using scale param; we assume normalized here.
    d = depth.astype(np.float32)
    eps = 1e-6
    inv = 1.0 / (d + eps)             # inverse-depth
    inv = np.log(inv + 1.0)           # log scaling
    
    # pseudo-height: normalize relative to image bottom (approximate ground at bottom rows)
    h, w = d.shape
    ys = np.arange(h).reshape(h,1)
    # height proxy: pixels nearer bottom are lower; use depth to modulate
    height = ( (h - ys) / h ) * d     # crude height proxy
    # normal-like: use sobel on depth
    dx = cv2.Sobel(d, cv2.CV_32F, 1, 0, ksize=3)
    dy = cv2.Sobel(d, cv2.CV_32F, 0, 1, ksize=3)
    # normal magnitude (edge strength)
    nm = np.sqrt(dx*dx + dy*dy)
    # stack and normalize to 0-1
    out = np.stack([height, inv, nm], axis=2)
    # normalize channelwise
    for c in range(3):
        ch = out[:,:,c]
        ch = ch - ch.min()
        if ch.max() > 0:
            ch = ch / (ch.max()+1e-9)
        out[:,:,c] = ch
    out = (out * 255).astype(np.uint8)
    return out


In [ ]:
# Cell 3
class DualSnowpoleDataset(Dataset):
    def __init__(self, images_dir, range_dir, labels_dir, img_size=1024, transforms=None):
        self.images_dir = Path(images_dir)
        self.range_dir = Path(range_dir)
        self.labels_dir = Path(labels_dir)
        self.paths = sorted([p for p in self.images_dir.glob("*.png")])
        self.img_size = img_size
        self.transforms = transforms
    
    def __len__(self):
        return len(self.paths)
    
    def load_labels(self, stem):
        p = self.labels_dir / f"{stem}.txt"
        boxes = []
        if p.exists():
            with open(p) as f:
                for line in f:
                    vals = line.strip().split()
                    if len(vals) >= 5:
                        cls = int(vals[0])
                        nums = list(map(float, vals[1:5]))
                        boxes.append([cls]+nums)
        return np.array(boxes, dtype=np.float32)  # Nx5
    
    def __getitem__(self, idx):
        img_path = self.paths[idx]
        stem = img_path.stem
        # read color
        bgr = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        bgr = cv2.resize(bgr, (self.img_size, self.img_size))
        pseudo = make_pseudocolor(bgr)  # HxWx3 uint8
        
        # read range
        rpath = self.range_dir / f"{stem}.png"
        if not rpath.exists():
            rpath = self.range_dir / f"{stem}.jpg"
        depth = cv2.imread(str(rpath), cv2.IMREAD_GRAYSCALE)
        if depth is None:
            depth = np.zeros((self.img_size, self.img_size), dtype=np.uint8)
        depth = cv2.resize(depth, (self.img_size, self.img_size))
        # normalize depth to [0,1] float for compute_hha
        depth_f = depth.astype(np.float32) / 255.0
        hha = compute_hha(depth_f)  # HxWx3 uint8
        
        # convert to CHW tensors float32 in [0,1]
        pa = torch.from_numpy(pseudo.transpose(2,0,1).astype(np.float32)/255.0).float()
        hb = torch.from_numpy(hha.transpose(2,0,1).astype(np.float32)/255.0).float()
        
        labels = self.load_labels(stem)  # Nx5 (cls,x,y,w,h)
        return pa, hb, labels, str(img_path)
    
def collate_fn(batch):
    # batch is list of tuples: pa, hb, labels, path
    pas, hbs, labels_list, paths = zip(*batch)
    pas = torch.stack(pas)  # [B,3,H,W]
    hbs = torch.stack(hbs)  # [B,3,H,W]
    return pas, hbs, labels_list, paths


In [ ]:
# Cell 4
class SimpleEncoder(nn.Module):
    def __init__(self, in_channels, widths=[32,64,128]):
        super().__init__()
        layers = []
        c = in_channels
        for w in widths:
            layers.append(nn.Conv2d(c, w, 3, stride=2, padding=1))
            layers.append(nn.BatchNorm2d(w))
            layers.append(nn.SiLU())
            c = w
        # output channels = last width
        self.net = nn.Sequential(*layers)
        self.out_channels = widths[-1]
    def forward(self, x):
        return self.net(x)  # [B,C,H/8,W/8] if 3 strides of 2

# Channel & spatial attention (CBAM-lite)
class ChannelSpatialAttention(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels//reduction, 1),
            nn.SiLU(),
            nn.Conv2d(channels//reduction, channels, 1),
            nn.Sigmoid()
        )
        self.spatial = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, groups=channels),
            nn.Sigmoid()
        )
    def forward(self, x):
        ca = self.mlp(x) * x
        sa = self.spatial(x) * x
        return ca + sa

# Cross-modal attention fusion (simplified CMX)
class CrossModalFusion(nn.Module):
    def __init__(self, c_rgb, c_depth, out_c):
        super().__init__()
        self.rgb_proj = nn.Conv2d(c_rgb, out_c, 1)
        self.dep_proj = nn.Conv2d(c_depth, out_c, 1)
        # gating
        self.gate = nn.Sequential(
            nn.Conv2d(out_c*2, out_c, 1),
            nn.SiLU(),
            nn.Conv2d(out_c, out_c, 1),
            nn.Sigmoid()
        )
        self.att = ChannelSpatialAttention(out_c)
    def forward(self, fr, fd):
        # fr, fd: [B,C,H,W] (should be same spatial size)
        R = self.rgb_proj(fr)
        D = self.dep_proj(fd)
        M = torch.cat([R, D], dim=1)
        g = self.gate(M)
        fused = g * R + (1.0 - g) * D
        fused = self.att(fused)
        return fused

# Tweaking layer: a light "detail-preserving" block
class TweakBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.conv1 = nn.Conv2d(c, c, 3, padding=1, dilation=1)
        self.conv2 = nn.Conv2d(c, c, 3, padding=2, dilation=2)  # dilated helps capture elongated objects
        self.bn = nn.BatchNorm2d(c)
        self.act = nn.SiLU()
        self.att = ChannelSpatialAttention(c)
    def forward(self, x):
        y = self.act(self.bn(self.conv1(x) + self.conv2(x)))
        y = self.att(y)
        return x + y

# Simple YOLO-like Head (single-scale for clarity; you can extend to multi-scale)
class SimpleYoloHead(nn.Module):
    def __init__(self, in_channels, num_anchors=3, num_classes=1):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, in_channels, 3, padding=1)
        self.pred = nn.Conv2d(in_channels, num_anchors * (5 + num_classes), 1)
        self.num_anchors = num_anchors
        self.num_classes = num_classes
    def forward(self, x):
        x = F.silu(self.conv(x))
        out = self.pred(x)
        # reshape to [B, num_anchors, 5+num_classes, H, W]
        B, C, H, W = out.shape
        out = out.view(B, self.num_anchors, 5 + self.num_classes, H, W)
        return out

# Full model combining the pieces
class DualFusionDetector(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()
        # Branch A: pseudo-color
        self.enc_a = SimpleEncoder(3, widths=[32,64,128])
        # Branch B: HHA/dep
        self.enc_b = SimpleEncoder(3, widths=[16,32,64])  # lighter
        # Fuse to a common channel
        self.fuse = CrossModalFusion(self.enc_a.out_channels, self.enc_b.out_channels, out_c=128)
        # tweak
        self.tweak = TweakBlock(128)
        # neck: upsample small -> higher resolution (we keep simple)
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        # head
        self.head = SimpleYoloHead(in_channels=128, num_anchors=3, num_classes=num_classes)
    def forward(self, a, b):
        # a: [B,3,H,W]; b: [B,3,H,W]
        fa = self.enc_a(a)  # [B,C1,H/8,W/8]
        fb = self.enc_b(b)  # [B,C2,H/8,W/8]
        fused = self.fuse(fa, fb)  # [B,128,H/8,W/8]
        tweaked = self.tweak(fused)
        up = self.up(tweaked)  # [B,128,H/4,W/4] -> helps small objects
        preds = self.head(up)  # [B,na,5+cls,H/4,W/4]
        return preds


In [ ]:
# Cell 5
# simple box IoU for tensors (xyxy)
def tensor_iou(box1, box2):
    # box1: [N,4] x1y1x2y2 ; box2: [M,4]
    N = box1.shape[0]; M = box2.shape[0]
    lt = torch.max(box1[:,None,:2], box2[None,:,:2])  # [N,M,2]
    rb = torch.min(box1[:,None,2:], box2[None,:,2:])  # [N,M,2]
    wh = (rb - lt).clamp(min=0)
    inter = wh[:,:,0] * wh[:,:,1]
    area1 = (box1[:,2]-box1[:,0])*(box1[:,3]-box1[:,1])
    area2 = (box2[:,2]-box2[:,0])*(box2[:,3]-box2[:,1])
    union = area1[:,None] + area2[None,:] - inter
    return inter / (union + 1e-9)

def ciou_loss(pred_boxes, target_boxes):
    # pred_boxes, target_boxes: [K,4] xyxy; return mean (1 - IoU)
    iou = tensor_iou(pred_boxes, target_boxes).diag()
    return (1 - iou).mean()

# anchors simple heuristic: we will set anchors to some aspect ratios tuned to narrow tall boxes
# For now use fixed anchors (w,h normalized) e.g. tiny narrow anchors
default_anchors = torch.tensor([[0.02,0.6],[0.03,0.8],[0.015,0.4]])  # relative (w,h) for H/4xW/4 grid


In [ ]:
# Cell 6
def decode_preds(preds, anchors, stride, conf_thresh=0.01):
    # preds: [B,na,5+cls,H,W]
    B, na, nc, H, W = preds.shape
    num_cls = nc - 5
    preds = preds.permute(0,1,3,4,2).contiguous()  # [B,na,H,W,5+cls]
    p = preds.sigmoid()  # treat all as sigmoid for simplicity
    # grid
    device = preds.device
    ys, xs = torch.meshgrid(torch.arange(H, device=device), torch.arange(W, device=device), indexing='ij')
    xs = xs.float(); ys = ys.float()
    out_boxes = []
    out_scores = []
    for b in range(B):
        boxes_b = []
        scores_b = []
        for a in range(na):
            pa = p[b,a]  # H W (5+cls)
            conf = pa[...,4]
            mask = conf > conf_thresh
            if mask.sum() == 0:
                continue
            tx = pa[...,0][mask]; ty = pa[...,1][mask]
            tw = pa[...,2][mask]; th = pa[...,3][mask]
            cx = xs[mask]; cy = ys[mask]
            # convert to normalized xywh relative to full image: (cx+tx)/W etc then to xyxy
            cxn = (cx + tx) / W
            cyn = (cy + ty) / H
            anchor = anchors[a].to(device)
            w_rel = (anchor[0] * torch.exp(tw))  # anchor scaling trick
            h_rel = (anchor[1] * torch.exp(th))
            x1 = cxn - w_rel/2; y1 = cyn - h_rel/2
            x2 = cxn + w_rel/2; y2 = cyn + h_rel/2
            cls_probs = pa[...,5:]  # sigmoid
            cls_scores, cls_idx = torch.max(cls_probs, dim=-1)
            score = conf * cls_scores
            for i in range(len(score)):
                boxes_b.append([x1[i].item(), y1[i].item(), x2[i].item(), y2[i].item()])
                scores_b.append(score[i].item())
        out_boxes.append(np.array(boxes_b) if boxes_b else np.zeros((0,4)))
        out_scores.append(np.array(scores_b) if scores_b else np.zeros((0,)))
    return out_boxes, out_scores


In [ ]:
# Cell 7
# config paths - change to your dataset
ROOT = Path("SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions/SnowPole_Detection_Dataset")
IMAGES_TRAIN = ROOT / "combined_color/train"   # pseudo-color source (will be processed via make_pseudocolor)
RANGE_TRAIN = ROOT / "range/train"
LABELS_TRAIN = ROOT / "labels/train"
IMAGES_VAL = ROOT / "combined_color/valid"
RANGE_VAL = ROOT / "range/valid"
LABELS_VAL = ROOT / "labels/valid"

train_ds = DualSnowpoleDataset(IMAGES_TRAIN, RANGE_TRAIN, LABELS_TRAIN, img_size=1024)
val_ds   = DualSnowpoleDataset(IMAGES_VAL, RANGE_VAL, LABELS_VAL, img_size=1024)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, collate_fn=collate_fn, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=2)

model = DualFusionDetector(num_classes=1).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)

anchors = default_anchors  # tensor [[w,h],...], normalized to image

# simple training loop (1 epoch demo)
model.train()
for epoch in range(1):  # extend epochs as needed
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}")
    for pa, hb, labels_list, paths in pbar:
        pa = pa.to(device)
        hb = hb.to(device)
        preds = model(pa, hb)  # [B,na,5+cls,H',W']
        # decode and compute toy loss via matching:
        # For each batch element, we will assign GTs to the highest-overlap predicted center cell anchor (very simplified)
        loss_obj = torch.tensor(0.0, device=device)
        loss_box = torch.tensor(0.0, device=device)
        loss_cls = torch.tensor(0.0, device=device)
        B = pa.shape[0]
        for i in range(B):
            gt = labels_list[i]  # Nx5
            if gt.shape[0] == 0:
                # encourage no-object -> push objectness low (skip for simplicity)
                continue
            # decode preds to boxes for this sample (use decode preds with low threshold)
            out_boxes, out_scores = decode_preds(preds[i:i+1], anchors, stride=None, conf_thresh=0.001)
            pred_boxes = out_boxes[0]  # Nx4 numpy
            if pred_boxes.shape[0] == 0:
                # heavy penalty if no predictions
                loss_obj = loss_obj + 1.0
                continue
            # convert GT to xyxy
            gt_xyxy = xywh_to_xyxy(gt[:,1:5]).astype(np.float32)
            # match greedily by IoU
            pb = torch.from_numpy(pred_boxes).to(device).float()
            tb = torch.from_numpy(gt_xyxy).to(device).float()
            # compute pairwise IoU and pick best matches (toy)
            if pb.shape[0] == 0 or tb.shape[0] == 0:
                continue
            ious = tensor_iou(pb, tb)  # [P,G]
            # take diagonal of best matches by P->G
            max_iou_per_pred, idx = ious.max(dim=1)
            # box loss = (1 - IoU) mean for matched ones
            loss_box = loss_box + (1 - max_iou_per_pred).mean()
            # objectness loss: encourage matched preds to have objectness high (we use preds sigmoid earlier)
            # Extract objectness predictions at those positions (this is messy in simplified head)
            loss_obj = loss_obj + ((1 - max_iou_per_pred).mean())
            # classification loss is trivial for single class domain; skip or use BCE on class score
        total_loss = loss_box + 0.1*loss_obj + 0.1*loss_cls
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        pbar.set_postfix(loss=total_loss.item())


In [ ]:
# Cell 8
def evaluate_map50(model, dataloader, iou_thresh=0.5, conf_thresh=0.05):
    model.eval()
    all_scores = []
    all_labels = []
    all_gt_counts = 0
    with torch.no_grad():
        for pa, hb, labels_list, paths in tqdm(dataloader, desc="Eval"):
            pa = pa.to(device); hb = hb.to(device)
            preds = model(pa, hb)
            boxes_batch, scores_batch = decode_preds(preds, anchors, stride=None, conf_thresh=conf_thresh)
            B = len(boxes_batch)
            for i in range(B):
                pred_boxes = boxes_batch[i]
                pred_scores = scores_batch[i]
                gt = labels_list[i]
                gt_boxes = xywh_to_xyxy(gt[:,1:5]) if len(gt)>0 else np.zeros((0,4))
                all_gt_counts += len(gt_boxes)
                # For PR curve gather each pred as TP or FP at IoU >= thresh
                # Simple greedy matching
                matched = np.zeros(len(gt_boxes), dtype=bool)
                for s, pb in sorted(zip(pred_scores, pred_boxes), key=lambda x: -x[0]):
                    if len(gt_boxes)==0:
                        all_scores.append((s, 0))  # FP
                        continue
                    ious = []
                    for g in gt_boxes:
                        # coords likely normalized; ensure same range
                        ious.append(iou_xyxy(pb, g.reshape(1,4))[0])
                    ious = np.array(ious)
                    mi = ious.max()
                    arg = ious.argmax()
                    if mi >= iou_thresh and not matched[arg]:
                        matched[arg] = True
                        all_scores.append((s, 1))
                    else:
                        all_scores.append((s, 0))
    # compute precision-recall
    if len(all_scores)==0:
        return 0.0
    all_scores = sorted(all_scores, key=lambda x: -x[0])
    tp_cum = np.cumsum([x[1] for x in all_scores])
    fp_cum = np.cumsum([1-x[1] for x in all_scores])
    rec = tp_cum / (all_gt_counts + 1e-9)
    prec = tp_cum / (tp_cum + fp_cum + 1e-9)
    ap = voc_ap(rec, prec)
    return ap

# run (single epoch)
ap50 = evaluate_map50(model, val_loader)
print("mAP@0.5 (toy):", ap50)
